# Einsatzdaten der Berliner Feuerwehr

## Einleitung und Fragestellung

git clone https://github.com/Berliner-Feuerwehr/BF-Open-Data

Wird die Einhaltung der Hilfsfrist bei Einsätzen der Berliner Feuerwehr stärker durch
- die Art des Einsatzes,
- die Systemlast der Leitstelle
- dem Standort (nach Berliner Bezirken)
- oder externe Faktoren wie das Wetter beeinflusst?

Abschluss:
- Kooperation mit KV Berlin
- was muss geschehen was kann besser
Hypothese: Das Wetter beeinflusst die Auslastung der Leitstellen, beides beeinflusst die Einhaltung der Hilfsfristen

Open Meteo weil REST-API + JSON, DWD hat nur CSV

Zunächst wurden die CSV dateien gesichet. Dann claude coding hinzugezogen was spannend wäre, dann analysiert

Leistellen erst ab 2024 verfügbar. Analyse dadurch eingeschränkt

### Hypothesen

1. Die Art des Einsatzes (z.B. kritischer Rettungsdienst vs. Brand) beeinflusst die Einhaltung der Hilfsfrist unterschiedlich stark — kritische Einsätze werden priorisiert und sollten häufiger eingehalten werden.
2. Eine hohe Systemlast der Leitstelle (viele unbeantwortete Anrufe, lange Antwortzeiten) korreliert mit einer schlechteren Hilfsfrist-Einhaltung.
3. Extreme Wetterlagen (Stürme, Starkregen, Hitze) erhöhen sowohl die Systemlast der Leitstelle als auch die Wahrscheinlichkeit, dass die Hilfsfrist verfehlt wird — Wetter wirkt also vermutlich indirekt über die Leitstelle.

Einschränkung: Leitstellen-/Anrufdaten sind erst ab 2024 verfügbar. Der gemeinsame Datensatz für die Modellierung ist deshalb auf 2024–2025 beschränkt, auch wenn Einsatz- und Wetterdaten weiter zurückreichen.

### Wichtige Korrektur: keine gesetzliche Hilfsfrist in Berlin

Berlin hat **keine gesetzlich bindende Hilfsfrist**, sondern ein politisch vereinbartes **"Schutzziel"**. Quellen nennen dafür unterschiedliche Zahlen: Drittquellen sprechen von 8 Minuten in 75% der Fälle (Schutzklasse A), die Berliner Feuerwehr selbst (Open-Data-Seite) nennt **90% Erreichungsgrad in 10 bzw. 11 Minuten, je nach Notfallart**.

> "Berlin [hat] statt einer Hilfsfrist, sog. Schutzziele vereinbart" — [vdek.com](https://www.vdek.com/LVen/BERBRA/fokus/reform-der-notfallversorgung.html)

> "Laut Schutzziel muss in Berlin der Rettungsdienst in 90% aller Einsätze in weniger als 10 Minuten bzw. 11 Minuten, je nach Art des Notfalls, mit mindestens einem Einsatzmittel vor Ort sein." — [berliner-feuerwehr.de](https://www.berliner-feuerwehr.de/service/open-data/)

**Methodische Entscheidung:** Wir bleiben trotzdem bei **8 Minuten (480 Sekunden)** als Schwelle für unsere Zielvariable `hilfsfrist` — nicht weil das Berlins offizielles Schutzziel ist (das wäre eher 10/11 Minuten), sondern weil **8 Minuten die medizinisch begründete Grenze ist** (z.B. bei Kreislauf-/Atemstillstand, wo irreversible Hirnschäden nach wenigen Minuten ohne Sauerstoffversorgung beginnen). Diese Grenze ist nicht verhandelbar, im Gegensatz zu einem politisch festgelegten Schutzziel-Wert. Unsere Zahlen sind dadurch **strenger** als Berlins eigener (ohnehin schon weicher) Maßstab — das heißt: die in dieser Analyse gefundenen Verfehlungsraten sind eine Untergrenze für das tatsächliche Problem, nicht übertrieben.

Eigene kritische Einordnung (unbelegt, aber naheliegend): Es ist denkbar, dass politische Schutzziel-Werte (egal ob 75%/8min oder 90%/10-11min) bewusst an das anpassen, was gerade noch erreichbar ist, statt am medizinischen Bedarf. Das ändert nichts an unserer Entscheidung, die medizinische 8-Minuten-Grenze als Maßstab zu verwenden.

# Datenimport
Laden von den täglichen Missionsdaten, den täglichen Anrufdaten sowie den Einsatzdaten
Überführen in Datenformate für die spätere Analyse
Start- und Enddatum für die Analyse manuell setzen um Covid-19 Ausreißer rauszunehmen

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

#set start and end date for analysis
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2025-12-31")

#daily mission data
df_daily_mission = pd.read_csv("data/BF-Open-Data/Datasets/Daily_Data/BFw_mission_data_daily.csv")
#daily mission set in date format
df_daily_mission["mission_created_date"] = pd.to_datetime (df_daily_mission["mission_created_date"])
df_daily_mission = df_daily_mission.set_index("mission_created_date").sort_index()


#daily call data
df_daily_call = pd.read_csv("data/BF-Open-Data/Datasets/Daily_Data/BFw_call_data_daily.csv")
#daily call set to date format
df_daily_call["calls_received_day"] = pd.to_datetime(df_daily_call["calls_received_day"])
df_daily_call = df_daily_call.sort_values("calls_received_day").set_index("calls_received_day")
#call_count als str eingelesen → in int umwandeln
df_daily_call["call_count"] = pd.to_numeric(df_daily_call["call_count"], errors="coerce")

#mission data set
df_mission_data = pd.concat([
    pd.read_csv("data/BF-Open-Data/Datasets/Mission_Data/mission_data_set_open_data_2022.csv", low_memory=False),
    pd.read_csv("data/BF-Open-Data/Datasets/Mission_Data/mission_data_set_open_data_2023.csv", low_memory=False),
    pd.read_csv("data/BF-Open-Data/Datasets/Mission_Data/mission_data_set_open_data_2024.csv", low_memory=False),
    pd.read_csv("data/BF-Open-Data/Datasets/Mission_Data/mission_data_set_open_data_2025.csv", low_memory=False),], ignore_index=True)

#mission data set to date format
df_mission_data["mission_date"] = pd.to_datetime(df_mission_data["mission_date"])
df_mission_data = df_mission_data.set_index("mission_date").sort_index()

#Filter for datas
df_daily_mission = df_daily_mission.loc[start_date:end_date]
df_daily_call = df_daily_call.loc[start_date:end_date]
df_mission_data = df_mission_data.loc[start_date:end_date]

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2022-01-01",
	"end_date": "2025-12-31",
	"daily": ["temperature_2m_max", "weather_code", "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"],
	"timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_weather_code = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(3).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["weather_code"] = daily_weather_code
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["snowfall_sum"] = daily_snowfall_sum
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

df_weather = pd.DataFrame(data = daily_data)
df_weather = df_weather.set_index("date")
df_weather.index = df_weather.index.tz_localize(None)
print("\nDaily data\n", df_weather)



# Datenverständnis

Überblick über die Daten verschaffen, um herauszufinden wie viele Daten vorhanden sind

In [ ]:
# mission data
print(df_mission_data.shape)
print(df_daily_mission.dtypes)
df_mission_data.head()

#missing values
df_mission_data.isnull().sum()

#zielvariable prüfen
print(df_mission_data["response_time"].describe())
print((df_mission_data["response_time"] <= 480).mean())

Über 2 Millionen Einsätze, 16 Variabeln

8-Minuten-Schutzziel (480 Sekunden) — keine gesetzliche Hilfsfrist, sondern ein politisch vereinbartes Schutzziel mit 75%-Zielquote (siehe Korrektur oben):
Mean: 677 Sekunden (ca 11 Minuten)
Median: 625 Sekunden (ca 10 Minuten)
Min: 121 Sekunden
Max: 1919 Sekunden (ca 32 Minuten)
Count: ca 264.000 Einträge haben keinen Wert (NaN) --> ca 12% muss bereinigt werden
17,2% der Einsätze halten das 8-Minuten-Schutzziel ein


In [ ]:
#daily mission data
print(df_daily_mission.shape)
print(df_daily_mission.dtypes)
df_daily_mission.head()

#missing values
df_daily_mission.isnull().sum()

#zielvariable prüfen
print(df_daily_mission["response_time_ems_critical_mean"].describe())
print(df_daily_mission["response_time_ems_critical_cpr_mean"].describe())

Critical EMS:
- Mean: 642 Sekunden (ca. 10.7 Minuten)
- Min: 558 Sekunden (ca. 9.3 Minuten) !!!

Critical EMS CPR:
- Mean: 505 Sekunden (ca. 8.4 Minuten)
- Min: 360 Sekunden (6 Minuten)

CPR Einsätze werden schneller beantwortet als allgemein kritische Fälle. Annahme: Priorisierungssystem der Leistelle funktioniert. CPR hat höchste prio.

In [ ]:
#daily calls
print(df_daily_call.shape)
print(df_daily_call.dtypes)
df_daily_call.head()

#missing values
df_daily_call.isnull().sum()

#zielvariable prüfen
print(df_daily_call["answer_time_mean_112"].describe())
print(df_daily_call["call_count_112"].describe())
print(df_daily_call["call_count_112_not_answered"].describe())
print(df_daily_call["call_count_112_answered_after_60_seconds"].describe())
print(df_daily_call["call_count_112_answered_within_10_seconds"].describe())

#Für die Systemauslastung
#Anteil nicht beantworteter Anrufe
#call_count_112 zählt laut Datenbeschreibung nur beantwortete Anrufe → korrekter Nenner: call_count_112 + call_count_112_not_answered
print("not answered rate")
df_daily_call["not_answered_rate"] = df_daily_call["call_count_112_not_answered"] / (df_daily_call["call_count_112"] + df_daily_call["call_count_112_not_answered"])
print(df_daily_call["not_answered_rate"].describe())

#Anteil nach 60 Sekunden beantwortet
print("after 60 rate")
df_daily_call["after_60s_rate"] = df_daily_call["call_count_112_answered_after_60_seconds"] / df_daily_call["call_count_112"]
print(df_daily_call["after_60s_rate"].describe())

In [ ]:
from scipy import stats

#Ausreißer mit Z score herausfinden (claude gefragt)
z_score = stats.zscore(df_daily_call["not_answered_rate"])
ausreisser = df_daily_call[abs(z_score) > 3]
print(ausreisser.sort_values("not_answered_rate", ascending = False))

Die Z-Score-Ausreißer (`|z|>3`) bei `not_answered_rate` und `after_60s_rate` lassen sich größtenteils erklären:

- **01.01.2025**: laut README-Disclaimer der Berliner Feuerwehr gab es zum Jahreswechsel 24/25 eine technische Einschränkung der 112-Rufnummer, Notrufe liefen über ein Fallback-System ohne automatisierte Erfassung. Die Zahlen für diesen Tag sind unvollständig → **wird aus der Analyse entfernt**, da es sich um einen Datenfehler statt um eine echte Lastspitze handelt.
- **23.06. und 26.06.2025**: an beiden Tagen gab es laut Open-Meteo-Daten schwere Gewitter mit Windböen von 85–88 km/h (weather_code 95/96) nach einer Hitzewelle. Das erklärt den Anstieg unbeantworteter Notrufe plausibel und **belegbar** → bleibt in den Daten, dient sogar als Beispiel für den vermuteten Wetter→Leitstelle-Zusammenhang.
- **07.09. und 20.11.2024**: keine Wetterauffälligkeiten an diesen Tagen. Mögliche Erklärung: Großveranstaltungen (07.09. Start von Lollapalooza Berlin, 20.11. diverse Konzerte) könnten zu mehr Notrufen geführt haben — diese Erklärung ist aber **nicht belegt**, da in einer Stadt mit 3,8 Mio. Einwohnern an fast jedem Tag irgendwelche Veranstaltungen stattfinden. Die Tage bleiben in den Daten, werden aber als ungeklärt dokumentiert statt kausal erklärt.

FEHLER MEINERSEITS (aus dem vorherigen Durchgang): die `call_count`-Spalten zählen nur die *beantworteten* Anrufe, nicht alle eingegangenen — das hat die ursprüngliche Berechnung der Rate verzerrt, ist hier aber bereits korrigiert (Nenner = `call_count_112 + call_count_112_not_answered`).

Antwortzeit wird von der Björn Steigerstiftung für kritische fälle
answer_time_mean_112:
- Mean: 26 Sekunden
- Max: 68 Sekunden

not_answered_rate:
mean 34,6%
Max 121% (?)

after_60s_rate:
- mean 13,2%
- max 40,4%


In [ ]:
#Wetterdaten
print(df_weather.shape)
print(df_weather.dtypes)
df_weather.head()

#missing values
print(df_weather.isnull().sum())

#zielvariable prüfen
print(df_weather.describe())

# Datenbereinigung

**Zeitraum:** Mission- und Wetterdaten reichen weiter zurück (2018 bzw. 2022), aber `df_daily_call` existiert erst seit 2024. Da die Leitstellen-Auslastung zentral für unsere Fragestellung ist, beschränken wir den gemeinsamen Datensatz für die Modellierung auf **2024–2025**. Explorative Auswertungen, die nur Mission- oder Wetterdaten brauchen, können trotzdem den längeren Zeitraum nutzen.

**Wetterdaten-Bug:** Durch die UTC→Europe/Berlin-Konvertierung landet der Zeitindex von `df_weather` auf 23:00 Uhr des Vortags statt auf Mitternacht (z.B. `2021-12-31 23:00:00` statt `2022-01-01`). Vor dem Join wird der Index korrigiert (+1 Stunde, dann auf den Tag normalisiert), sonst würden Wetterdaten dem falschen Tag zugeordnet.

**response_time NaN:** ca. 12% der Einsätze (≈265.000 von 1,86 Mio.) haben keinen Wert bei `response_time` — vermutlich weil nicht alle relevanten Zeitstempel vorlagen (siehe Felddokumentation: `..._timegoal_computed`). Diese Zeilen werden entfernt, da sich für sie keine Zielvariable berechnen lässt.

**response_time Ausreißer werden NICHT entfernt:** Ein Z-Score-Test zeigt ca. 36.000 Einsätze (~2%) mit ungewöhnlich langer Reaktionszeit (1455–1919 Sekunden). Das sind keine Messfehler (keine negativen/unplausiblen Werte), sondern genau die Fälle, in denen das Schutzziel verfehlt wurde — also der Kern unserer Fragestellung. Sie zu entfernen würde die Zielvariable künstlich verzerren.

**dispatchcode_criticality / dispatchcode_category NaN:** ca. 111.000 Einsätze haben keinen Dispatchcode (vermutlich Einsatztypen wie "Sonstige" oder "Notverlegung", die nicht über das normale Disponierungsschema laufen). Statt diese Zeilen zu droppen (zu hoher Datenverlust), wird eine eigene Kategorie "unbekannt" eingeführt.

**Zielvariable:** `hilfsfrist = response_time <= 480` (480 Sekunden = Berlins 8-Minuten-Schutzziel, **keine gesetzliche Hilfsfrist** — siehe Korrektur weiter oben in der Einleitung. Das offizielle Schutzziel selbst gilt für 75% der Fälle, nicht für 100%).

**Zeitliche Features:** aus `mission_date` werden Wochentag und Monat extrahiert. Eine Uhrzeit-Variable ("Stunde") ist nicht möglich, da `mission_date` in allen Jahresdateien (2018–2026) ausschließlich das Datum ohne Uhrzeitkomponente enthält — keiner der verfügbaren BF-Open-Data-Datensätze (Mission_Data, Daily_Data, Turnout_Times, KV_Data, Regional_Data) liefert eine Uhrzeit-Auflösung.


In [ ]:
#Wetter anpassen, Zeitstempel-Bug korrigieren
df_weather.index = (df_weather.index + pd.Timedelta(hours=1)).normalize()

#Kontrolle
print(df_weather.index.min(), df_weather.index.max())
print(df_weather.head())
print(df_weather.tail())

In [ ]:
# response_time NaN entfernen
print("vorher:", df_mission_data.shape)
#dropna löscht Zeilen wo NaN ist
df_mission_data = df_mission_data.dropna(subset=["response_time"])
print("nachher", df_mission_data.shape)

In [ ]:
# Zielvariable Hilfsfrist eingehalten
df_mission_data["hilfsfrist"] = df_mission_data["response_time"] <= 480
print(df_mission_data["hilfsfrist"].value_counts(normalize=True))

# Wochentag und Monat aus dem Datums-Index rausziehen
df_mission_data["wochentag"] = df_mission_data.index.day_name()
df_mission_data["monat"] = df_mission_data.index.month

print(df_mission_data[["wochentag", "monat"]].head())

### Kategoriale Variablen encodieren

Modelle rechnen nur mit Zahlen, nicht mit Text wie `"Brand"` oder `"Mitte"`.

- **NaN bei `dispatchcode_criticality`** → wird zu eigener Kategorie `"unbekannt"` (statt Zeilen zu löschen).
- **One-Hot-Encoding** für `mission_type`, `mission_location_district`, `dispatchcode_criticality`: jede Kategorie wird eine eigene 0/1-Spalte. So entsteht keine künstliche Rangordnung zwischen den Kategorien (z.B. Bezirken), wie es bei einer einfachen Durchnummerierung der Fall wäre.
- Ergebnis: mehr Spalten, gleich viele Zeilen.

In [ ]:
# NaN bei Dispatchcode als eigene Kategorie "unbekannt"
df_mission_data["dispatchcode_criticality"] = df_mission_data["dispatchcode_criticality"].fillna("unbekannt")

# One-Hot-Encoding der nominal kategorialen Variabeln
df_mission_data = pd.get_dummies(
    df_mission_data,
    columns=["mission_type", "mission_location_district", "dispatchcode_criticality"],
    prefix=["typ", "district", "criticality"],
)

print(df_mission_data.shape)
df_mission_data.head()

# Nach Rettungsdienst-Einsätze filtern für Hilfsfrist (inkl. Mischtyp mit Technischer Hilfeleistung, da beide AMPDS-Dispatchcodes haben)
print("vorher: ", df_mission_data.shape)
df_mission_data = df_mission_data[
    (df_mission_data["typ_Rettungsdienst"] == True) |
    (df_mission_data["typ_Rettungsdienst mit Technischer Hilfeleistung"] == True)
]
print("nachher:", df_mission_data.shape)

In [ ]:
# Silvester-Tag entfernen: dokumentierter Systemausfall (README-Disclaimer), keine echte Lastspitze
print("vorher:", df_daily_call.shape)
df_daily_call = df_daily_call.drop(pd.Timestamp("2025-01-01"))
print("nachher:", df_daily_call.shape)

# Explorative Analyse: Vorgehen

Wir schauen uns die Daten in vier Stufen an, statt direkt mit komplexen Visualisierungen zu starten:

1. **Langer Zeitraum (2022–2025), einzeln:** Einsatztyp, Wetter, Wochentag, Monat — je für sich gegen `hilfsfrist`. Zeigt schnell, welche Variablen überhaupt einen Effekt haben.
2. **Langer Zeitraum, kombiniert:** z.B. Heatmap Wochentag × Monat — nur für Variablen, die in Stufe 1 einzeln schon ein Muster zeigen.
3. **Kurzer Zeitraum (2024–2025), einzeln:** Leitstellen-Variablen (`not_answered_rate`, `answer_time_mean_112`, ...) gegen `hilfsfrist`/`response_time`. Geht nicht über den langen Zeitraum, da Leitstellendaten erst ab 2024 existieren.
4. **Kurzer Zeitraum, kombiniert:** Leitstelle zusammen mit Wetter bzw. Einsatztyp — testet direkt Hypothese 3 (wirkt Wetter indirekt über die Leitstelle auf die Hilfsfrist?).

In [ ]:
# Hilfsfrist-Quote pro Kritikalitätsstufe (statt Einsatztyp, da jetzt nur noch Rettungsdienst)
crit_spalten = [col for col in df_mission_data.columns if col.startswith("criticality_")]
quote_je_crit = {
    col.replace("criticality_", ""): df_mission_data.loc[df_mission_data[col], "hilfsfrist"].mean()
    for col in crit_spalten
}
quote_je_crit = pd.Series(quote_je_crit)

# Nach AMPDS-Dringlichkeit sortieren (O < A < B < C < D < E), statt nach Quote
reihenfolge = ["O", "A", "B", "C", "D", "E", "unbekannt"]
quote_je_crit = quote_je_crit.reindex([k for k in reihenfolge if k in quote_je_crit.index])

print(quote_je_crit)

quote_je_crit.plot(kind="bar")
plt.ylabel("Eingehaltungsrate")
plt.xlabel("Dringlichkeitsstufen")
plt.title("Einhaltung der Hilfsfrist im Rettungsdienst")
plt.xticks(rotation=0)

# Legende/Erklärung der AMPDS-Stufen als Bildunterschrift
plt.figtext(
    0.5, -0.05,
    "AMPDS-Dringlichkeitsstufen (steigend): O=Omega (niedrig) < A=Alpha < B=Bravo < C=Charlie < D=Delta < E=Echo (lebensbedrohlich)",
    wrap=True, ha="center", fontsize=8
)

plt.tight_layout()
plt.show()

## Wetter vs. Hilfsfrist (Stufe 1, einzeln)

Einsatzdaten (nur Rettungsdienst, langer Zeitraum 2022–2025) wurden auf das Tagesdatum mit den Wetterdaten gejoint. Für `temperature_2m_max`, `precipitation_sum` und `wind_speed_10m_max` wurde die Verteilung getrennt nach `hilfsfrist` (eingehalten/nicht eingehalten) verglichen.

**Befund:** Die Verteilungen von Temperatur, Niederschlag und Windgeschwindigkeit sind für beide Gruppen (Hilfsfrist eingehalten vs. nicht eingehalten) nahezu identisch — kein sichtbarer univariater Effekt.

**Einordnung:** Das bedeutet nicht, dass Wetter keine Rolle spielt, sondern dass es **isoliert betrachtet** keinen klaren direkten Zusammenhang mit der Hilfsfrist zeigt. Das passt zur dritten Hypothese, wonach Wetter eher *indirekt über die Systemlast der Leitstelle* wirkt, statt direkt auf die einzelne Anfahrtszeit. Dieser indirekte Pfad wird erst in Stufe 3/4 mit den echten Leitstellendaten (2024–2025) geprüft.

In [ ]:
import seaborn as sns
# Einsatzdaten mit Wetter joinen
df_weather_mission = df_mission_data.join(df_weather, how="left")

print(df_weather_mission.shape)
print(df_weather_mission[["temperature_2m_max", "precipitation_sum", "wind_speed_10m_max"]].isnull().sum())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

variabeln = ["temperature_2m_max", "precipitation_sum", "wind_speed_10m_max"]
titel = ["Temperatur (°C)", "Niederschlag (mm)", "Windgeschwindigkeit (km/h)"]

for ax, var, t in zip(axes, variabeln, titel):
    sns.boxplot(data=df_weather_mission, x="hilfsfrist", y=var, ax=ax)
    ax.set_title(t)
    ax.set_xlabel("Hilsfrist eingehalten")

plt.tight_layout()
plt.show()

In [ ]:
# Hilfsfrist-Quote pro Monat
quote_monat = df_mission_data.groupby("monat")["hilfsfrist"].mean()

quote_monat.plot(kind="bar")
plt.ylabel("Hilfsfrist eingehalten")
plt.title("Hilfsfrist eingehalten nach Monat")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Hilfsfrist-Quote je Wochentag
quote_wochentag = df_mission_data.groupby("wochentag")["hilfsfrist"].mean()
sort_wochentag = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
quote_wochentag = quote_wochentag.reindex(sort_wochentag)

quote_wochentag.plot(kind="bar")
plt.ylabel("Hilfsfrist eingehalten")
plt.title("Hilfsfrist eingehalten nach Wochentag")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Heatmap Hilfsfrist nach Wochentag, Monat
pivot = df_mission_data.pivot_table(index="wochentag", columns="monat", values="hilfsfrist", aggfunc="mean")

sort_wochentag = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = pivot.reindex(sort_wochentag)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Hilfsfrist eingehalten nach Wochentag und Monat")
plt.xlabel("Monat")
plt.ylabel("Wochentag")
plt.tight_layout()
plt.show()

## Wochentag & Monat vs. Hilfsfrist (Stufe 1 + 2)

**Monat (Stufe 1, einzeln):** Die Hilfsfrist-Quote schwankt nur minimal über die Monate — kein erkennbares saisonales Muster. Bestätigt den Befund aus dem direkten Wetter-Vergleich: Wetter/Saison zeigt keinen klaren Effekt auf die Hilfsfrist.

**Wochentag (Stufe 1, einzeln):** Am Wochenende wird die Hilfsfrist etwas häufiger eingehalten als werktags — der bisher einzige klare Unterschied in Stufe 1. Mögliche Erklärung: weniger Berufsverkehr und ggf. geringere Einsatzlast am Wochenende, was zur Leitstellen-Hypothese passt.

**Heatmap Wochentag × Monat (Stufe 2, kombiniert):** Nur geringe Abweichungen zwischen den Zellen, kein zusätzlicher Interaktionseffekt über die einzelnen Wochentags- und Monatsmuster hinaus. Der Wochenend-Effekt bleibt sichtbar, wird aber nicht durch die Jahreszeit verstärkt oder abgeschwächt.

**Fazit Stufe 1+2:** Von den getesteten Proxy-Variablen für den langen Zeitraum (2022–2025) zeigt nur der Wochentag ein nennenswertes Muster. Das spricht dafür, mit Stufe 3 (echte Leitstellendaten, 2024–2025) zu prüfen, ob sich die Wochenend-Beobachtung durch eine tatsächlich geringere Systemlast der Leitstelle erklären lässt.

In [ ]:
# Einsatzdaten auf erfassung der Anrufe begrenzen (Jahre) 2024-2025
df_mission_data_cut = df_mission_data.loc[pd.Timestamp("2024-01-01"):pd.Timestamp("2025-12-31")]

# Einsatzdaten joinen mit Anrufdaten
df_leitstelle = df_mission_data_cut.join(df_daily_call, how="left")

print(df_leitstelle.shape)
print(df_leitstelle[["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]].isnull().sum())

# Boxplot: Leistelle VS Hilfsfrist
df_leitstelle_plot = df_leitstelle.reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

leitstellen_var = ["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]
titel = ["Anteil nicht beantwortet", "Anteil nach 60s beantwortet", "Mittlere Antwortzeit (s)"]

for ax, var, t in zip(axes, leitstellen_var, titel):
    sns.boxplot(data=df_leitstelle_plot, x="hilfsfrist", y=var, ax=ax)
    ax.set_title(t)
    ax.set_xlabel("Hilfsfrist eingehalten")

plt.tight_layout()
plt.show()



In [ ]:
# Auf Tagesebene aggregieren statt pro Einsatz (vermeidet Verdünnung durch Einsatzebene-Varianz)
df_tage = df_leitstelle.groupby(df_leitstelle.index)["hilfsfrist"].mean().to_frame("hilfsfrist_quote")
df_tage = df_tage.join(df_daily_call[["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]])

print(df_tage.corr()["hilfsfrist_quote"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, var in zip(axes, ["not_answered_rate", "after_60s_rate", "answer_time_mean_112"]):
    ax.scatter(df_tage[var], df_tage["hilfsfrist_quote"], alpha=0.5)
    ax.set_xlabel(var)
    ax.set_ylabel("Hilfsfrist-Quote (Tag)")
plt.tight_layout()
plt.show()

In [ ]:
# Einsatzaufkommen pro Tag (Anzahl Rettungsdienst-Einsätze) als direkterer Systemlast-Indikator
df_tage["einsatz_anzahl"] = df_leitstelle.groupby(df_leitstelle.index).size()

print(df_tage.corr()["hilfsfrist_quote"])

# Zusätzlich: hängt not_answered_rate selbst mit dem Einsatzaufkommen zusammen?
print()
print("Korrelation not_answered_rate <-> Einsatzanzahl:", df_tage["not_answered_rate"].corr(df_tage["einsatz_anzahl"]))

## Leitstelle: Entscheidung, nicht weiter zu verfolgen

**Befund:** `not_answered_rate` korreliert statistisch signifikant mit der täglichen Hilfsfrist-Quote (r=-0,29, p≈0,000000, n=730 Tage). Der Zusammenhang ist real, aber moderat (erklärt nur ~9% der Varianz) — deutlich schwächer als Kritikalität oder Bezirk.

**Einschränkung:** Die Feldbeschreibung definiert `call_count_112_not_answered` nur als "Number of emergency calls (112) not answered", ohne zu klären, ob Mehrfachanrufe zum selben Vorfall (bei Notfällen typisch: mehrere Personen rufen denselben Unfall an) mitgezählt werden. Damit ist unklar, ob die Metrik wirklich "Systemüberlastung der Leitstelle" misst oder teilweise Anrufverhalten widerspiegelt. Die erklärbaren Ausreißertage (Silvester-Systemausfall, Sturmtage) sprechen für echte Aussagekraft, die unerklärten Tage (z.B. Lollapalooza-Vermutung) bleiben aber unbelegt.

**Entscheidung:** Aufgrund dieser Mess-Unsicherheit verfolgen wir die Leitstellen-Variablen für das Modelltraining **nicht weiter** — nicht weil kein Zusammenhang besteht (der ist statistisch belegt), sondern weil wir nicht sauber genug einordnen können, was genau gemessen wird. Bleibt als offene Frage für eine mögliche Folgeanalyse (z.B. Rücksprache mit der Leitstelle, was als "nicht beantwortet" gilt) dokumentiert, statt einen mehrdeutigen Befund ins Modell zu übernehmen.

In [ ]:
# Wie weit wird die Hilfsfrist bei Echo-Einsätzen über/unterschritten?
echo_response = df_mission_data.loc[df_mission_data["criticality_E"], "response_time"]

print(echo_response.describe())
print()
print("Median Überschreitung (nur verfehlte Fälle):", (echo_response[echo_response > 480] - 480).median(), "Sekunden")
print("Mittlere Überschreitung (nur verfehlte Fälle):", (echo_response[echo_response > 480] - 480).mean(), "Sekunden")

plt.figure(figsize=(8, 5))
plt.hist(echo_response, bins=50)
plt.axvline(480, color="red", linestyle="--", label="Hilfsfrist (480s)")
plt.xlabel("Response Time (Sekunden)")
plt.ylabel("Anzahl Einsätze")
plt.title("Verteilung der Reaktionszeit bei Echo-Einsätzen")
plt.legend()
plt.tight_layout()
plt.show()

## Hilfsfrist nach Bezirk

Aus Schritt 5 des Fahrplans nachgeholt: Hilfsfrist-Quote je Bezirk, über den gesamten Rettungsdienst-Datensatz (2022–2025).

In [ ]:
# Hilfsfrist-Quote nach Bezirk (alle Bezirke)
district_spalten = [col for col in df_mission_data.columns if col.startswith("district_")]
quote_je_bezirk = {
    col.replace("district_", ""): df_mission_data.loc[df_mission_data[col], "hilfsfrist"].mean()
    for col in district_spalten
}
quote_je_bezirk = pd.Series(quote_je_bezirk).sort_values(ascending=False)

print(quote_je_bezirk)
print()
print("Spannweite (max-min):", quote_je_bezirk.max() - quote_je_bezirk.min())

plt.figure(figsize=(10, 5))
quote_je_bezirk.plot(kind="bar")
plt.ylabel("Anteil Hilfsfrist eingehalten")
plt.title("Hilfsfrist-Einhaltung nach Bezirk (Rettungsdienst)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Hilfsfrist-Quote nach Bezirk x Kritikalitätsstufe
crit_spalten = [col for col in df_mission_data.columns if col.startswith("criticality_")]
district_spalten = [col for col in df_mission_data.columns if col.startswith("district_")]

pivot_daten = []
for d_col in district_spalten:
    bezirk = d_col.replace("district_", "")
    zeile = {}
    for c_col in crit_spalten:
        crit = c_col.replace("criticality_", "")
        maske = df_mission_data[d_col] & df_mission_data[c_col]
        zeile[crit] = df_mission_data.loc[maske, "hilfsfrist"].mean()
    pivot_daten.append(pd.Series(zeile, name=bezirk))

pivot_bezirk_crit = pd.DataFrame(pivot_daten)

# Spalten nach AMPDS-Dringlichkeit sortieren
reihenfolge = ["O", "A", "B", "C", "D", "E", "unbekannt"]
pivot_bezirk_crit = pivot_bezirk_crit[[c for c in reihenfolge if c in pivot_bezirk_crit.columns]]
# Zeilen nach Gesamtquote sortieren
pivot_bezirk_crit = pivot_bezirk_crit.loc[pivot_bezirk_crit.mean(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10, 7))
sns.heatmap(pivot_bezirk_crit, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Hilfsfrist-Quote nach Bezirk × Dringlichkeitsstufe")
plt.xlabel("AMPDS-Kritikalität")
plt.ylabel("Bezirk")
plt.tight_layout()
plt.show()

# Zielvariable & Features finalisieren

Feature-Matrix nur aus den Variablen mit belegtem Effekt: Kritikalität (AMPDS) und Bezirk, plus Einsatztyp (Rettungsdienst vs. Mischtyp). Wochentag, Monat, Wetter und Leitstelle bewusst ausgeschlossen (siehe Begründungen oben).

In [ ]:
# Feature-Matrix X zusammenstellen (nur Kritikalität, Bezirk, Einsatztyp)
feature_spalten = (
    [c for c in df_mission_data.columns if c.startswith("criticality_")] +
    [c for c in df_mission_data.columns if c.startswith("district_")] +
    ["typ_Rettungsdienst", "typ_Rettungsdienst mit Technischer Hilfeleistung"]
)

X = df_mission_data[feature_spalten]
y = df_mission_data["hilfsfrist"]

print(X.shape, y.shape)
print(X.columns.tolist())

# Modelltraining

## Modell 1: Logistische Regression

### Variante A: ohne `class_weight` (Baseline)

In [ ]:
# Train/Test Split (80/20)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Anteil hilfsfrist=True (train):", y_train.mean())
print("Anteil hilfsfrist=True (test):", y_test.mean())

# Logistische Regression trainieren
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

print("Trainings-Accuracy:", log_reg.score(X_train, y_train))
print("Test-Accuracy:", log_reg.score(X_test, y_test))

y_pred = log_reg.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Variante B: mit `class_weight="balanced"`

In [ ]:
# Logistische Regression mit class_weight="balanced" (Vergleich zur unbalanced Version oben)
log_reg_balanced = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg_balanced.fit(X_train, y_train)

print("Trainings-Accuracy:", log_reg_balanced.score(X_train, y_train))
print("Test-Accuracy:", log_reg_balanced.score(X_test, y_test))

y_pred_balanced = log_reg_balanced.predict(X_test)
print(confusion_matrix(y_test, y_pred_balanced))
print(classification_report(y_test, y_pred_balanced))

## Modell 2: Random Forest

In [ ]:
# Random Forest trainieren
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Trainings-Accuracy:", rf.score(X_train, y_train))
print("Test-Accuracy:", rf.score(X_test, y_test))

y_pred_rf = rf.predict(X_test)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

## Modell 3: Random Forest Regression

**Entscheidung:** k-Means fällt raus. Begründung: unsere Feature-Matrix besteht nur aus One-Hot-Spalten (Kritikalität, Bezirk, Einsatztyp), für die euklidisches Clustering wenig Mehrwert über die bereits erstellte Bezirk×Kritikalität-Heatmap hinaus gebracht hätte.

Stattdessen: ein **Regressionsmodell**, das die tatsächliche `response_time` (in Sekunden) vorhersagt, statt nur Ja/Nein zur Hilfsfrist — beantwortet die Frage "wie viele Minuten wird der Rettungswagen bei dieser Kritikalität/diesem Bezirk/Einsatztyp ungefähr brauchen?".

In [ ]:
# Regressionsmodell: response_time (Sekunden) statt hilfsfrist (Ja/Nein) vorhersagen
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

y_minuten = df_mission_data["response_time"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_minuten, test_size=0.2, random_state=42
)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)

y_pred_r = rf_reg.predict(X_test_r)
print("Mittlerer absoluter Fehler (Sekunden):", mean_absolute_error(y_test_r, y_pred_r))
print("Mittlerer absoluter Fehler (Minuten):", mean_absolute_error(y_test_r, y_pred_r) / 60)

# Modellbewertung

## Vergleich Log. Regression vs. Random Forest (Klassifikation)

In [ ]:
# Accuracy, F1, Precision, Recall im direkten Vergleich
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

modelle = {
    "LogReg (unbalanced)": y_pred,
    "LogReg (balanced)": y_pred_balanced,
    "Random Forest": y_pred_rf,
}

vergleich = []
for name, pred in modelle.items():
    vergleich.append({
        "Modell": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (True)": precision_score(y_test, pred),
        "Recall (True)": recall_score(y_test, pred),
        "F1 (True)": f1_score(y_test, pred),
    })

df_vergleich = pd.DataFrame(vergleich).set_index("Modell")
print(df_vergleich)

## Random Forest: Feature Importances

In [ ]:
# Welche Features nutzt der Random Forest (Klassifikation) am meisten?
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

plt.figure(figsize=(10, 6))
importances.plot(kind="bar")
plt.ylabel("Wichtigkeit")
plt.title("Feature Importances — Random Forest (Hilfsfrist-Klassifikation)")
plt.tight_layout()
plt.show()

# Interpretation

**Welche Prädiktorgruppe erklärt am meisten?**

Modellvergleich (Test-Set):

| Modell | Accuracy | Precision (True) | Recall (True) | F1 (True) |
|---|---|---|---|---|
| LogReg (unbalanced) | 0,80 | 0,58 | **0,01** | 0,02 |
| LogReg (balanced) | 0,58 | 0,26 | 0,61 | **0,37** |
| Random Forest | 0,62 | 0,27 | 0,55 | 0,36 |

Random Forest und balancierte LogReg liegen mit F1≈0,36–0,37 etwa gleichauf — die unbalancierte LogReg ist mit Recall=0,01 wertlos für die eigentliche Fragestellung (erkennt fast keine eingehaltenen Hilfsfristen).

**Feature Importances (Random Forest):** die größten Gewichte liegen auf `criticality_D` (16,5%), `district_Friedrichshain-Kreuzberg` (15,5%), `district_Mitte` (9,7%) und `criticality_E` (7,2%) — Kritikalität und Bezirk dominieren beide, in etwa vergleichbarer Größenordnung. Das bestätigt quantitativ, was die EDA schon zeigte:

1. **Kritikalität (AMPDS)** — stärkster Einzeleffekt in der EDA (Quote steigt von 9,7% bei O auf 44,8% bei E).
2. **Bezirk** — fast ebenso wichtig im Modell, Spannweite 12,6–30,5% in der EDA.
3. **Bezirk × Kritikalität** — additiv, kein Bezirk durchbricht das Kritikalitätsmuster; bei Echo-Fällen ist die Bezirks-Lücke mit 40 Punkten am größten.
4. **Leitstelle** — real (p≈0), aber moderat (~9% erklärte Varianz) und methodisch unsicher; bewusst nicht ins Modell aufgenommen.
5. **Wochentag** — schwacher Nebeneffekt. **Wetter, Monat** — kein nachweisbarer Effekt.

**Wo versagt das Modell?**

- Mit nur drei groben Kategorien bleibt viel Restvarianz unerklärt (Verkehr, Tageszeit, Fahrzeugverfügbarkeit — nicht in den Daten enthalten).
- Klassenungleichgewicht (80/20) lässt die unbalancierte LogReg in die Falle laufen: hohe Accuracy täuscht, sie sagt fast immer "False" vorher.
- Mit Gewichtung wird die Vorhersage ehrlicher, aber F1≈0,36–0,37 bleibt weit von einem verlässlichen Einzelfall-Modell entfernt.
- Das Regressionsmodell (response_time direkt) hat einen mittleren Fehler von ~2,9 Minuten — brauchbar als grobe Schätzung pro Kategorie-Kombination, nicht für präzise Einzelfall-Aussagen.

# Reflexion & KV-Ausblick

**Grenzen der Analyse:**
- 8-Minuten-Schwelle ist medizinisch begründet, aber nicht Berlins offizielles Schutzziel (10/11 Min., 90%) — Zahlen nicht 1:1 mit offiziellen Statistiken vergleichbar.
- Leitstellen-Variablen methodisch unklar (Mehrfachanrufe pro Vorfall nicht ausschließbar) — deshalb nicht im Modell, echter Effekt evtl. unterschätzt.
- Nur 12 Bezirke als Geo-Auflösung; Planungsraum-Ebene zeigt deutlich größere Spannweite (72,7 Punkte) — räumliche Varianz im Modell wahrscheinlich unterschätzt.
- Kreislaufstillstand-Kategorie enthält vermutlich auch bereits verstorbene Personen — 55%-Verfehlungsquote bei Echo nicht 1:1 als "vermeidbare Todesfälle" interpretierbar.
- Datenzeiträume uneinheitlich: Mission/Wetter 2022–2025, Leitstelle nur 2024–2025 — keine vollständig deckungsgleiche Basis für alle drei Hypothesen.
- Nur Korrelationen geprüft, keine Kausalität (Sturmtage als einziges halbwegs kausales Beispiel).
- Einfache Modelle (3 Kategorien, keine Interaktionsterme); kein Gradient Boosting, keine Mehrebenen-Modelle getestet.
- Keine Uhrzeit-Variable möglich (Datenquelle liefert nur Tagesdatum) — Tageszeit-Effekte komplett unbeobachtbar.

**KV-Ausblick (offene Folgefrage):**
- Die Berliner Feuerwehr kann Einsätze, die laut Meldebild nicht zwingend den Rettungsdienst brauchen, an die Kassenärztliche Vereinigung (KV) Berlin abgeben — ein zusätzlicher, in dieser Analyse nicht ausgewerteter Datensatz (`daily_kv_data.csv`) existiert dafür.
- Ob und wie stark diese Abgabe die Leitstellen-Auslastung und damit die Hilfsfrist tatsächlich entlastet, ist mit den hier verwendeten Daten/Methoden nicht abschließend zu beantworten — dafür bräuchte es eine eigene, zeitlich differenziertere Untersuchung (z.B. wirkt eine Abgabe am Vormittag auf die Hilfsfrist am Nachmittag?).
- Bleibt als offene Frage für eine mögliche KV-Kooperation/Folgeanalyse stehen, statt hier eine unbelegte Wirkung zu behaupten.

# Quellen

https://rettungslandschaft.steiger-stiftung.de/notrufbearbeitung-und-hilfsfrist/
https://www.elab2go.de/demo-py/
https://open-meteo.com/en/docs/historical-forecast-api?start_date=2022-01-01&end_date=2025-12-31&timezone=Europe%2FBerlin&hourly=&daily=temperature_2m_max,weather_code,precipitation_sum,snowfall_sum,wind_speed_10m_max#hourly_weather_variables